### Approach
Initially, I used the full YCSEP dataset (where each row is a segment) to form embeddings and indexed each segments embeddings into a Chroma DB vector store. This was to find all audio segments that speaker X would appear in. This performed poorly because majority of segments were no longer than 2-3 seconds long, resulting in poor performance especially when using the 1 min audio clip as a reference. This process took too long with very poor accuracy. 

I switched strategies to first find out instances where speaker X appears in the whole podcast using a 1 min sliding window to embedding similarity with the reference audio. 

The threshold was originally set at 0.3 but i realised early on that this created a lot of false positive and set the threshold to 0.5 instead


### Analysing Results
After filtering out all audios with 1 min windows exceeding the 0.5 threshold. We had the following breakdown:

| Channel | Detections |
|---|---|
| Yah_Lah_BUT | 93 |
| The_Daily_Ketchup_Podcast | 11 |
| NOTG | 2 |
| YouGotWatch | 1 |

The highest matched podcast was **"The Woke Salaryman on the BIGGEST Problem Rich People Face"** from Yah_Lah_BUT with a cosine similarity of **0.6852**, matched at the 30–90s window.


Despite setting a higher threshold, it was clear from the set of podcasts above that there wasn't an overlap of speakers, there were still a lot of false positives. I decided to manually review the highest matched podcast and ascertain who speaker X was. I started with the Woke Salary Man podcast, and identified one of the two founders who sounded similar to the reference audio.


### Speaker X is *Goh Wei Choon* from the Woke Salary Man
I decided to investigate (manually) for the source of the reference audio on YouTube and was able to find the following audio clip with the exact audio as `speaker_ref.wav`. I investigated this video because the subject matter was the same as the reference audio.


[![Speaker X original video audio](https://img.youtube.com/vi/8uAPU0iBOsc/0.jpg)](https://youtu.be/8uAPU0iBOsc?si=5A8teOqBqhDcuQ0e&t=60)


The following code extracts podcast with a similarity of more than 0.5 and creates `detected_titles.txt`


In [ ]:
import json
from pathlib import Path
from urllib.parse import unquote

### Load embedding progress data

In [ ]:
BASE_DIR = Path(".").resolve()
PROGRESS_FILE = BASE_DIR / "data" / "embed_progress.json"

with open(PROGRESS_FILE) as f:
    progress = json.load(f)

total = len(progress)
matches = {k: v for k, v in progress.items() if v is not None}
no_match = total - len(matches)

print(f"Total podcasts processed: {total}")
print(f"Speaker X detected:       {len(matches)}")
print(f"No match:                 {no_match}")

### Extract YouTube titles from matched entries

In [ ]:
def extract_title(match_info: dict) -> str:
    """Extract the human-readable YouTube title from the wav_url."""
    url = match_info["wav_url"]
    filename = unquote(url.split("/")[-1]).replace(".wav", "")
    parts = filename.split("--", 2)
    return parts[2] if len(parts) > 2 else filename


def extract_video_id(podcast_key: str) -> str:
    """Extract the YouTube video ID from the podcast key."""
    parts = podcast_key.split("--", 2)
    return parts[1] if len(parts) > 1 else ""


results = []
for key, info in matches.items():
    results.append({
        "title": extract_title(info),
        "video_id": extract_video_id(key),
        "similarity": info["similarity"],
        "match_start": info["match_start"],
        "match_end": info["match_end"],
        "podcast_key": key,
    })

results.sort(key=lambda r: r["similarity"], reverse=True)
print(f"Extracted {len(results)} titles")

### Overview of detections by channel

In [ ]:
from collections import Counter

channels = Counter()
for key in matches:
    parts = key.split(".")
    channel = parts[1] if len(parts) >= 2 else "unknown"
    channels[channel] += 1

print("Detections by channel:")
print("-" * 40)
for channel, count in channels.most_common():
    print(f"  {channel:<35} {count:>3}")

### All detected videos (ranked by similarity)

In [ ]:
print(f"{'#':<4} {'Sim':>6}  {'Time':>14}  {'Title'}")
print("-" * 100)
for i, r in enumerate(results, 1):
    time_str = f"{r['match_start']:.0f}-{r['match_end']:.0f}s"
    print(f"{i:<4} {r['similarity']:>6.4f}  {time_str:>14}  {r['title']}")

### Save detected titles to file

In [ ]:
output_path = BASE_DIR / "detected_titles.txt"

titles = [r["title"] for r in results]

with open(output_path, "w", encoding="utf-8") as f:
    for title in titles:
        f.write(title + "\n")

print(f"Saved {len(titles)} titles to {output_path}")

### Generate detected timestamps for Speaker X segments

I curated the podcasts that Wei Choon appeared in and identified which `speaker` he was in the `YCSEP_static.csv` dataset, I then filtered out all his segments into `detected_timestamps.json` using the following code.

In [ ]:
import csv

CSV_PATH = BASE_DIR.parent / "asr" / "data" / "YCSEP_static.csv"
TIMESTAMPS_OUTPUT = BASE_DIR / "detected_timestamps.json"

targets = [
    {
        "file_prefix": "20211224--MNFbu28ZWmM--Brains behind The Woke Salaryman",
        "speaker": "SPEAKER_00",
        "title": None,
    },
    {
        "file_prefix": "20220726--7xS2y_U0Bz4--The Woke Salaryman on the BIGGEST",
        "speaker": "SPEAKER_02",
        "title": None,
    },
]

segments_map = {0: [], 1: []}

with open(CSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        for i, t in enumerate(targets):
            if row["file"].startswith(t["file_prefix"]) and row["speaker"] == t["speaker"]:
                if t["title"] is None:
                    raw = row["file"].replace(".TextGrid", "")
                    parts = raw.split("--", 2)
                    t["title"] = parts[2] if len(parts) > 2 else raw
                segments_map[i].append({
                    "start_time": float(row["start_time"]),
                    "end_time": float(row["end_time"]),
                    "text": row["text"],
                })

result = []
for i, t in enumerate(targets):
    segs = sorted(segments_map[i], key=lambda s: s["start_time"])
    result.append({"title": t["title"], "segments": segs})

with open(TIMESTAMPS_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

for r in result:
    print(f"{r['title']}: {len(r['segments'])} segments")
print(f"\nSaved to {TIMESTAMPS_OUTPUT}")